# Notebook 32 - Second corpus, part 1: IoMT bridge, taxonomy, teachers, alert-semantic graph

**Corpus.** CIC-IoMT-2024 (WiFi/MQTT), loaded through the IoMT compression project's own loader so the SABER arm inherits that project's preprocessing exactly (train-fitted 0.1/99.9 clipping, then standardisation). Only the official CIC train side is opened; exact duplicates are removed; a 20% validation partition is carved out stratified by attack type with seed 42. The official CIC test parquet is never opened in any SABER arm. The clip bounds and scaler parameters needed to process it identically later are stored in the bridge manifest.

**Stage 2 is a `%%writefile`** that creates `src/saber/bridge_iomt.py`. Run it once; it is part of the commit. It reads the IoMT project's Category-A leakage audit and applies it only if the file's format is unambiguous, recording what it did in the manifest either way.

**What carries over unchanged.** `LabelTaxonomy`, the three cost profiles, `full_model_audit`, and the `risk_graph` builder with the frozen CICIoT2023 configuration. The families map to the same lowercase names the taxonomy requires.

**Teachers.** Both CICIoT2023 topologies, trained under the NB29 acceptance rule (benign escalation at most 10%, attack recall at least 90%, fine macro-F1 at least 0.50), with the class-weight exponent chosen from three candidates; every candidate is cached so nothing is lost on interruption, and a failure to meet the rule is recorded, not hidden.

**Stages.** 1 bootstrap, 2 bridge module, 3 pre-registration, 4 build and audit the split (tensors cached under `data/iomt_bridge/`, gitignored, hashed in the manifest), 5 models and helpers, 6 teachers, 7 the graph, 8 record and figure. GPU required for stage 6.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, hashlib, copy
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
assert REPO.exists(), f"SABER repo not found: {REPO}"
assert IOMT_REPO.exists(), f"IoMT repo not found: {IOMT_REPO}"
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "32_iomt_bridge"; OUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"; DATA_CACHE.mkdir(parents=True, exist_ok=True)   # gitignored tensors
MODEL_DIR = REPO / "models/iomt"; MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("SABER repo:", REPO, "| IoMT repo:", IOMT_REPO, "| device:", DEVICE)


In [ ]:
%%writefile src/saber/bridge_iomt.py
"""Bridge from the CIC-IoMT-2024 frozen split (IoMT compression project) into the SABER pipeline.

Contract mirrors bridge_ciciot.load_bridge: returns (train_loader, val_loader, class_names, taxonomy,
manifest). The official CIC test parquet is never opened here; the clip bounds and scaler parameters
needed to process it identically are stored in the manifest for a later locked audit.
"""
from __future__ import annotations

import hashlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from src.saber.taxonomy import LabelTaxonomy

IOMT_SRC = "/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research/src"
IOMT_REPORTS = "/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research/reports"
FAMILY_LOWER = {"Benign": "benign", "DDoS": "ddos", "DoS": "dos", "MQTT": "mqtt",
                "Recon": "recon", "Spoofing": "spoofing"}
SEED = 42
VAL_FRACTION = 0.20


def _sha256_array(a: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()


def iomt_taxonomy(class_names, family_by_class) -> LabelTaxonomy:
    fams = tuple(FAMILY_LOWER[f] for f in family_by_class)
    benign = [c for c, f in zip(class_names, fams) if f == "benign"]
    assert len(benign) == 1, f"expected exactly one benign class, got {benign}"
    return LabelTaxonomy(class_names=tuple(class_names), family_by_class=fams,
                         benign_class=benign[0], dataset="CIC-IoMT-2024")


def _leakage_drop_list() -> tuple[list[str], str]:
    """Apply the IoMT project's Category-A leakage audit only if its format is unambiguous."""
    path = os.path.join(IOMT_REPORTS, "cat_a_feature_leakage.csv")
    if not os.path.exists(path):
        return [], "leakage report absent; no feature dropped"
    rep = pd.read_csv(path)
    feat_col = next((c for c in rep.columns if c.lower() in ("feature", "column", "name")), None)
    flag_col = next((c for c in rep.columns if c.lower() in ("drop", "flagged", "flag", "leak", "leaky", "exclude")), None)
    if feat_col is None or flag_col is None:
        return [], f"leakage report present but not applied (columns {list(rep.columns)})"
    flags = rep[flag_col]
    if flags.dtype == bool or set(flags.dropna().unique()) <= {0, 1, True, False, "True", "False"}:
        drop = rep.loc[flags.astype(str).isin(["True", "1"]), feat_col].astype(str).tolist()
        return drop, f"dropped {len(drop)} Category-A features per {os.path.basename(path)}[{flag_col}]"
    return [], f"leakage report present but flag column {flag_col!r} not boolean; not applied"


def build_bridge(cache_dir: str | Path, batch_size: int = 1024, force: bool = False):
    """Load, deduplicate, split, preprocess and cache. Returns (train_loader, val_loader, class_names, taxonomy, manifest)."""
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = cache_dir / "manifest.json"
    tensors_path = cache_dir / "tensors.pt"
    if manifest_path.exists() and tensors_path.exists() and not force:
        manifest = json.loads(manifest_path.read_text())
        blob = torch.load(tensors_path, map_location="cpu", weights_only=False)
    else:
        if IOMT_SRC not in sys.path:
            sys.path.insert(0, IOMT_SRC)
        import gc
        import iomtc_data as D  # the IoMT project's loader, used as-is

        try:
            train_df, _test_unused = D.load_official(add_targets=True)
        except Exception as exc:                                     # the loader resolves paths from its own config
            hint = getattr(D, "_official_dir", lambda: "?")()
            raise RuntimeError(f"IoMT loader failed (resolved data dir: {hint}); run from a runtime where "
                               f"{IOMT_SRC} imports and its data directory exists") from exc
        del _test_unused                                             # never touched in the SABER arms
        gc.collect()
        feat_cols = [c for c in D.feature_columns(train_df)]
        drop, leak_note = _leakage_drop_list()
        feat_cols = [c for c in feat_cols if c not in set(drop)]
        non_numeric = [c for c in feat_cols if not np.issubdtype(train_df[c].dtype, np.number)]
        assert not non_numeric, f"non-numeric feature columns would corrupt preprocessing: {non_numeric}"

        n_raw = int(len(train_df))
        train_df = train_df.drop_duplicates(subset=feat_cols + ["attack_type"]).reset_index(drop=True)
        n_dedup = int(len(train_df))

        # frozen stratified validation carve-out from the official train side
        rng = np.random.default_rng(SEED)
        val_mask = np.zeros(n_dedup, dtype=bool)
        for _, idx in train_df.groupby("attack_type").indices.items():
            idx = np.asarray(idx); rng.shuffle(idx)
            val_mask[idx[: int(round(VAL_FRACTION * len(idx)))]] = True

        class_names = sorted(train_df["attack_type"].astype(str).unique().tolist())
        fam_of = train_df.drop_duplicates("attack_type").set_index("attack_type")["family"].astype(str).to_dict()
        family_by_class = [fam_of[c] for c in class_names]
        cls_index = {c: i for i, c in enumerate(class_names)}
        y_all = train_df["attack_type"].astype(str).map(cls_index).to_numpy(np.int64)

        tr_df, va_df = train_df[~val_mask], train_df[val_mask]
        del train_df; gc.collect()
        los, his = D.fit_clip(tr_df, feat_cols)
        X_tr = D.apply_clip(tr_df, feat_cols, los, his)
        del tr_df; gc.collect()
        X_va = D.apply_clip(va_df, feat_cols, los, his)
        del va_df; gc.collect()
        mean, std = X_tr.mean(axis=0), X_tr.std(axis=0)
        std = np.where(std < 1e-8, 1.0, std)
        X_tr = ((X_tr - mean) / std).astype(np.float32)
        X_va = ((X_va - mean) / std).astype(np.float32)
        y_tr, y_va = y_all[~val_mask], y_all[val_mask]
        gc.collect()

        blob = {"X_tr": torch.from_numpy(X_tr), "y_tr": torch.from_numpy(y_tr),
                "X_va": torch.from_numpy(X_va), "y_va": torch.from_numpy(y_va)}
        torch.save(blob, tensors_path)
        manifest = {
            "dataset": "CIC-IoMT-2024 WiFi/MQTT, official CIC train side; official test never opened",
            "n_official_train_rows": n_raw, "n_after_exact_dedup": n_dedup,
            "dedup_rule": "exact duplicates on all retained features plus attack_type",
            "val_fraction": VAL_FRACTION, "split_seed": SEED,
            "n_train": int(len(y_tr)), "n_val": int(len(y_va)),
            "feature_columns": feat_cols, "n_features": len(feat_cols), "leakage_note": leak_note,
            "clip_percentiles": [0.1, 99.9], "clip_low": los.tolist(), "clip_high": his.tolist(),
            "scaler_mean": mean.tolist(), "scaler_std": std.tolist(),
            "class_names": class_names, "family_by_class": family_by_class,
            "train_class_counts": np.bincount(y_tr, minlength=len(class_names)).tolist(),
            "val_class_counts": np.bincount(y_va, minlength=len(class_names)).tolist(),
            "sha256_X_tr": _sha256_array(X_tr), "sha256_y_tr": _sha256_array(y_tr),
            "sha256_X_va": _sha256_array(X_va), "sha256_y_va": _sha256_array(y_va),
        }
        manifest_path.write_text(json.dumps(manifest, indent=2))

    class_names = manifest["class_names"]
    taxonomy = iomt_taxonomy(class_names, manifest["family_by_class"])
    gen = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(TensorDataset(blob["X_tr"], blob["y_tr"]), batch_size=batch_size, shuffle=True, generator=gen)
    val_loader = DataLoader(TensorDataset(blob["X_va"], blob["y_va"]), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, class_names, taxonomy, manifest


In [ ]:
# Stage 3 - pre-registration (frozen before any teacher exists)
PREREG = {
    "arm": "B_second_corpus_bridge",
    "corpus": ("CIC-IoMT-2024 WiFi/MQTT, loaded through the IoMT compression project's own loader; "
               "official CIC train side only, exact-duplicate rows removed, 20% stratified validation "
               "carve-out (seed 42); official CIC test parquet is never opened in any SABER arm"),
    "preprocessing": "train-fitted 0.1/99.9 percentile clipping then train-fitted standardisation, "
                     "identical to the IoMT compression project",
    "hierarchy": "fine = attack_type (Benign plus attack types), family = the IoMT project's six families",
    "cost_profiles": "the three CICIoT2023 profiles, unchanged; they are stances an operator might hold "
                     "and are the authors' construction, aggregated by CVaR exactly as before",
    "graph": "built from the shallow teacher's validation logits with the frozen CICIoT2023 risk_graph "
             "configuration (top_k 3, confusion 0.5 / margin 0.5, min support 20, CVaR q 0.75)",
    "teachers": {"architectures": "CNN1D two-block (64/128) and DeepCNN1D four-block, the CICIoT2023 topologies",
                 "recipe": "Adam 1e-3, batch 1024, inverse-power class weighting with exponent chosen from "
                           "{0.25, 0.35, 0.50}, 4 fixed epochs, seed 0",
                 "acceptance": {"benign_to_attack_rate": "<= 0.10 on validation",
                                "binary_attack_recall": ">= 0.90 on validation",
                                "fine_macro_f1": ">= 0.50 on validation"},
                 "rule": "highest fine macro-F1 among candidates meeting all three; if none meets them, "
                         "the lowest-benign-escalation candidate with attack recall >= 0.90 is used and "
                         "teacher_meets_prereg is recorded False"},
    "evaluation_subsample": "validation rows capped at 4,000 per class, shuffled with seed 12345",
    "no_test_access": True,
}
(OUT / "B_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))


In [ ]:
# Stage 4 - build the bridge (cached after the first run) and audit the split
import importlib, src.saber.bridge_iomt as BI
importlib.reload(BI)
TRAIN_LOADER, VAL_LOADER, CLASS_NAMES, taxonomy, MANIFEST = BI.build_bridge(DATA_CACHE)
N_CLASSES = len(CLASS_NAMES)
print("features:", MANIFEST["n_features"], "|", MANIFEST["leakage_note"])
print("official train rows:", MANIFEST["n_official_train_rows"], "-> after dedup:", MANIFEST["n_after_exact_dedup"],
      "| SABER train:", MANIFEST["n_train"], "| SABER val:", MANIFEST["n_val"])
print("classes:", N_CLASSES, "| families:", list(taxonomy.families), "| benign:", taxonomy.benign_class)
counts = pd.DataFrame({"class": CLASS_NAMES, "family": MANIFEST["family_by_class"],
                       "train": MANIFEST["train_class_counts"], "val": MANIFEST["val_class_counts"]})
counts.to_csv(OUT / "class_counts.csv", index=False); print(counts.to_string(index=False))
(OUT / "bridge_manifest.json").write_text(json.dumps(
    {k: v for k, v in MANIFEST.items() if k not in ("clip_low", "clip_high", "scaler_mean", "scaler_std")}, indent=2))
assert MANIFEST["n_val"] > 0 and N_CLASSES >= 10


In [ ]:
# Stage 5 - models, evaluation subsample, helpers
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit


class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


ARCHS = {"shallow": CNN1D, "deep": DeepCNN1D}

Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]
BENIGN_IDX = taxonomy.benign_index
print("evaluation rows:", len(_idx), "| benign share:", round(float((EX_Y == BENIGN_IDX).mean()), 4))

_train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
COUNTS = np.bincount(_train_y, minlength=N_CLASSES)


def class_weights(alpha):
    w = np.zeros(N_CLASSES, dtype=np.float64); nz = COUNTS > 0
    w[nz] = 1.0 / np.power(COUNTS[nz], alpha); w[nz] /= w[nz].mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def forward_logits(model, X=None):
    X = EX_X if X is None else X
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def audit_logits(lg, y=None):
    a = full_model_audit(lg, EX_Y if y is None else y, taxonomy, DEFAULT_COST_PROFILES)
    return {k: float(v) for k, v in a.items()}


def train_epochs(model, epochs, weights, lr=1e-3):
    model = model.to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        model.train()
        for xb, yb in TRAIN_LOADER:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


def meets(row):
    return row["benign_to_attack_rate"] <= 0.10 and row["binary_attack_recall"] >= 0.90 and row["fine_macro_f1"] >= 0.50


In [ ]:
# Stage 6 - teachers under the pre-registered acceptance rule (per-candidate caching)
ALPHA_GRID = [0.25, 0.35, 0.50]; EPOCHS_TEACHER = 4
SEL = OUT / "teacher_selection.csv"
rows = pd.read_csv(SEL).to_dict("records") if SEL.exists() else []
TEACHERS, TEACHER_META = {}, {}
for arch, cls in ARCHS.items():
    final = MODEL_DIR / f"{arch}_teacher_seed0.pt"
    if final.exists():
        payload = torch.load(final, map_location="cpu", weights_only=False)
        m = cls(N_CLASSES); m.load_state_dict(payload["state_dict"]); TEACHERS[arch] = m.to(DEVICE).eval()
        TEACHER_META[arch] = {k: payload[k] for k in ("alpha", "meets_criteria")}
        print(f"{arch}: teacher loaded from cache (alpha={payload['alpha']}, meets={payload['meets_criteria']})"); continue
    for alpha in ALPHA_GRID:
        cand = MODEL_DIR / f"{arch}_candidate_alpha{alpha:.2f}.pt"
        if cand.exists() and any(r["architecture"] == arch and float(r["alpha"]) == alpha for r in rows):
            continue
        torch.manual_seed(0); np.random.seed(0)
        m = train_epochs(cls(N_CLASSES), EPOCHS_TEACHER, class_weights(alpha))
        a = audit_logits(forward_logits(m))
        row = {"architecture": arch, "alpha": alpha, "benign_to_attack_rate": a["benign_to_attack_rate"],
               "binary_attack_recall": a["binary_attack_recall"], "fine_macro_f1": a["fine_macro_f1"],
               "family_macro_f1": a["family_macro_f1"], "attack_to_benign_rate": a["attack_to_benign_rate"]}
        row["meets_criteria"] = bool(meets(row))
        rows = [r for r in rows if not (r["architecture"] == arch and float(r["alpha"]) == alpha)] + [row]
        torch.save({"state_dict": m.cpu().state_dict(), "alpha": alpha}, cand)
        pd.DataFrame(rows).to_csv(SEL, index=False)
        print(f"{arch} alpha={alpha}: b2a={row['benign_to_attack_rate']:.4f} attack_recall={row['binary_attack_recall']:.4f} "
              f"fine_f1={row['fine_macro_f1']:.4f} -> {'meets' if row['meets_criteria'] else 'does not meet'}")
    table = pd.DataFrame([r for r in rows if r["architecture"] == arch])
    ok = table[table["meets_criteria"].astype(str) == "True"]
    if len(ok):
        chosen = ok.sort_values("fine_macro_f1", ascending=False).iloc[0]; meets_flag = True
    else:
        viable = table[table["binary_attack_recall"] >= 0.90]; pool = viable if len(viable) else table
        chosen = pool.sort_values("benign_to_attack_rate").iloc[0]; meets_flag = False
        print(f"WARNING: no {arch} candidate met the acceptance rule; using the fallback and recording it")
    alpha = float(chosen["alpha"])
    m = cls(N_CLASSES); m.load_state_dict(torch.load(MODEL_DIR / f"{arch}_candidate_alpha{alpha:.2f}.pt",
                                                     map_location="cpu", weights_only=False)["state_dict"])
    torch.save({"state_dict": m.state_dict(), "alpha": alpha, "meets_criteria": meets_flag}, final)
    TEACHERS[arch] = m.to(DEVICE).eval(); TEACHER_META[arch] = {"alpha": alpha, "meets_criteria": meets_flag}

T_AUDIT = {}
for arch, m in TEACHERS.items():
    a = audit_logits(forward_logits(m)); T_AUDIT[arch] = a
    print(f"\n{arch} teacher:", {k: round(a[k], 4) for k in ["fine_macro_f1", "family_macro_f1", "benign_to_attack_rate",
                                                             "attack_to_benign_rate", "binary_attack_recall", "ece15"]})
json.dump({a: {**TEACHER_META[a], **T_AUDIT[a], "checkpoint": str((MODEL_DIR / f"{a}_teacher_seed0.pt").relative_to(REPO)),
               "sha256": hashlib.sha256((MODEL_DIR / f"{a}_teacher_seed0.pt").read_bytes()).hexdigest()}
           for a in TEACHERS}, open(OUT / "teacher_audit.json", "w"), indent=2)


In [ ]:
# Stage 7 - the alert-semantic vulnerability graph from the shallow teacher's validation logits
from src.saber.risk_graph import (aggregate_robust_edge_weights, build_alert_semantic_vulnerability_graph,
                                  merge_cost_profile_graphs, save_graph_bundle)
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml")); graph_cfg = SABER_CFG["risk_graph"]

VAL_LOGITS = forward_logits(TEACHERS["shallow"], Xv.to(DEVICE)); VAL_LABELS = VAL_Y_ALL
profile_graphs = []
for name, profile in DEFAULT_COST_PROFILES.items():
    profile_graphs.append(build_alert_semantic_vulnerability_graph(
        VAL_LOGITS, VAL_LABELS, taxonomy, profile,
        top_k=int(graph_cfg["top_k_competitors"]), temperature=float(graph_cfg["temperature"]),
        confusion_weight=float(graph_cfg["confusion_weight"]), margin_weight=float(graph_cfg["margin_weight"]),
        min_class_support=int(graph_cfg["min_class_support"]), environments=None))
edges_by_profile = merge_cost_profile_graphs(profile_graphs)
robust_graph = aggregate_robust_edge_weights(edges_by_profile, method=str(graph_cfg["robust_aggregation"]),
                                             q=float(graph_cfg["robust_cvar_q"]))
save_graph_bundle(OUT, edges_by_profile, robust_graph, {
    "dataset": taxonomy.dataset, "n_validation": int(len(VAL_LABELS)), "n_classes": taxonomy.n_classes,
    "families": list(taxonomy.families), "cost_profiles": list(DEFAULT_COST_PROFILES),
    "risk_graph_config": graph_cfg, "teacher": "shallow", "environment_count": None})
assert np.isclose(robust_graph["robust_weight"].sum(), 1.0)
print("edges:", len(robust_graph), "| sources covered:", robust_graph["source_index"].nunique(), "of", taxonomy.n_classes)
print(robust_graph.groupby("transition_type")["robust_weight"].sum().round(3).to_string())
print(robust_graph.nlargest(12, "robust_weight")[["source_class", "target_class", "transition_type", "robust_weight"]].round(4).to_string(index=False))


In [ ]:
# Stage 8 - record and figure
verdict = {
    "arm": "B_second_corpus_bridge",
    "bridge": {k: MANIFEST[k] for k in ["n_official_train_rows", "n_after_exact_dedup", "n_train", "n_val", "n_features", "leakage_note",
                                        "sha256_X_tr", "sha256_X_va"]},
    "n_classes": N_CLASSES, "families": list(taxonomy.families),
    "teachers": json.load(open(OUT / "teacher_audit.json")),
    "graph_edges": int(len(robust_graph)), "graph_sources_covered": int(robust_graph["source_index"].nunique()),
    "prereg": json.load(open(OUT / "B_PREREGISTRATION.json")),
}
(OUT / "B_bridge_verdict.json").write_text(json.dumps(verdict, indent=2))
print(json.dumps({k: v for k, v in verdict.items() if k != "prereg"}, indent=2))

fam_flow = robust_graph.groupby(["source_family", "target_family"])["robust_weight"].sum().unstack(fill_value=0.0)
fig, ax = plt.subplots(figsize=(6.2, 4.6))
im = ax.imshow(fam_flow.values, cmap="Reds"); ax.set_xticks(range(len(fam_flow.columns))); ax.set_xticklabels(fam_flow.columns, rotation=30, ha="right")
ax.set_yticks(range(len(fam_flow.index))); ax.set_yticklabels(fam_flow.index); ax.set_xlabel("absorber family"); ax.set_ylabel("source family")
ax.set_title("IoMT alert-semantic graph: family-level flow"); plt.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout(); fig.savefig(OUT / "B_family_flow.png", dpi=200); plt.show()
print("written ->", OUT)


In [ ]:
print(open("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research/reports/cat_a_feature_leakage.csv").read())

In [ ]:
%%bash
cd /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
cat ../.gitconfig > /root/.gitconfig
cat ../.git-credentials > /root/.git-credentials && chmod 600 /root/.git-credentials
python3 - <<'EOF'
import json
nb = json.load(open("notebooks/32_iomt_bridge.ipynb"))
for c in nb["cells"]:
    if c.get("cell_type") == "code":
        c["outputs"] = []; c["execution_count"] = None
json.dump(nb, open("/tmp/stripped.ipynb", "w"), ensure_ascii=False, indent=1)
EOF
blob=$(git hash-object -w /tmp/stripped.ipynb)
git update-index --add --cacheinfo 100644,$blob,notebooks/32_iomt_bridge.ipynb
git add results/saber/32_iomt_bridge
git commit -m "NB32 results: IoMT bridge, teachers and graph. Bridge reproduces the IoMT compression paper's canonical count exactly (7,160,831 official train rows -> 4,515,078 after exact dedup; SABER train 3,612,063 / val 903,015; 19 classes, 6 families; 45 features; leakage audit present but not applied because its disposition column is non-boolean, decision pending). Both teachers meet the pre-registered acceptance rule on every candidate; rule selects shallow alpha 0.5 (fine macro-F1 0.702, benign escalation 0.087, attack recall 0.982, ECE 0.044) and deep alpha 0.35 (0.736, 0.040, 0.972, ECE 0.061). Graph: 57 edges, 19 of 19 sources covered, cross-family attack mass 0.64 dominated by DoS/DDoS same-protocol pairs, attack-to-benign 0.16, benign-to-attack 0.08"
git push origin saber-ids-method
git log --oneline -1